In [1]:
import pandas as pd
import numpy as np

# pastikan string
df["tanggal"] = df["tanggal"].astype(str)
df["Waktu"] = df["Waktu"].astype(str)

# buat datetime
df["datetime"] = pd.to_datetime(df["tanggal"] + " " + df["Waktu"], errors="coerce")

# hapus yang gagal
df = df.dropna(subset=["datetime"])

# urutkan
df = df.sort_values(by=["station", "datetime"])

print("Sebelum resample:", len(df))

NameError: name 'df' is not defined

In [123]:
df = df.drop_duplicates(subset=["station", "datetime"])

In [124]:
df_list = []

for station in df["station"].unique():
    
    df_station = df[df["station"] == station].copy()
    
    df_station = df_station.set_index("datetime")
    
    # bikin semua jam
    df_station = df_station.resample("1h").asfreq()
    
    # isi kembali station
    df_station["station"] = station
    
    df_list.append(df_station)

# gabungkan
df = pd.concat(df_list).reset_index()

print("Setelah resample:", len(df))


Setelah resample: 191909


In [125]:
print("\nContoh data:")
print(df[df["station"] == df["station"].iloc[0]].head(30))


Contoh data:
              datetime  Waktu  ISPU PM10  ISPU PM2.5  ISPU SO2   ISPU CO  \
0  2022-01-12 17:00:00  17:00   0.194872    0.188612  0.510204  0.191781   
1  2022-01-12 18:00:00  18:00   0.194872    0.188612  0.510204  0.191781   
2  2022-01-12 19:00:00    NaN        NaN         NaN       NaN       NaN   
3  2022-01-12 20:00:00  20:00   0.194872    0.188612  0.510204  0.178082   
4  2022-01-12 21:00:00    NaN        NaN         NaN       NaN       NaN   
5  2022-01-12 22:00:00    NaN        NaN         NaN       NaN       NaN   
6  2022-01-12 23:00:00  23:00   0.194872    0.188612  0.510204  0.178082   
7  2022-01-13 00:00:00    NaN        NaN         NaN       NaN       NaN   
8  2022-01-13 01:00:00    NaN        NaN         NaN       NaN       NaN   
9  2022-01-13 02:00:00    NaN        NaN         NaN       NaN       NaN   
10 2022-01-13 03:00:00    NaN        NaN         NaN       NaN       NaN   
11 2022-01-13 04:00:00    NaN        NaN         NaN       NaN       NaN  

In [126]:
# ubah "-" jadi NaN
df.replace("-", np.nan, inplace=True)

# kolom polusi
kolom_polusi = [
    "ISPU PM10", 
    "ISPU PM2.5", 
    "ISPU SO2", 
    "ISPU CO", 
    "ISPU O3", 
    "ISPU NO2"
]

# pastikan numerik
for col in kolom_polusi:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# forward fill per station
df[kolom_polusi] = df.groupby("station")[kolom_polusi].ffill()

# fallback mean
df[kolom_polusi] = df[kolom_polusi].fillna(df[kolom_polusi].mean())

In [127]:
df["tanggal"] = df["datetime"].dt.date
df["Waktu"] = df["datetime"].dt.strftime("%H:%M")

df["tahun"] = df["datetime"].dt.year
df["bulan"] = df["datetime"].dt.month
df["jam"] = df["datetime"].dt.hour

In [128]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df[kolom_polusi] = scaler.fit_transform(df[kolom_polusi])

In [129]:
print("\nFinal data:")
print(df.head())


Final data:
             datetime  Waktu  ISPU PM10  ISPU PM2.5  ISPU SO2   ISPU CO  \
0 2022-01-12 17:00:00  17:00   0.194872    0.188612  0.510204  0.191781   
1 2022-01-12 18:00:00  18:00   0.194872    0.188612  0.510204  0.191781   
2 2022-01-12 19:00:00  19:00   0.194872    0.188612  0.510204  0.191781   
3 2022-01-12 20:00:00  20:00   0.194872    0.188612  0.510204  0.178082   
4 2022-01-12 21:00:00  21:00   0.194872    0.188612  0.510204  0.178082   

    ISPU O3  ISPU NO2 Status By PM10 Status By PM2.5  ... Status By CO  \
0  0.098361  0.037736           Baik          Sedang  ...         Baik   
1  0.103825  0.037736           Baik          Sedang  ...         Baik   
2  0.103825  0.037736            NaN             NaN  ...          NaN   
3  0.103825  0.042453           Baik          Sedang  ...         Baik   
4  0.103825  0.042453            NaN             NaN  ...          NaN   

  Status By O3 Status By NO2 Critical Parameter Overall ISPU Status  \
0         Baik      

In [130]:
import os
os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/final_polusi_clean.csv", index=False)

